In [ ]:
# Cell 0 — Environment + importability

from pathlib import Path  
import os  
import sys  
import json  
import pandas as pd  

# --- EDIT THESE 3 ONLY IF YOUR MACHINE PATHS DIFFER ---
REPO_ROOT = Path(r"C:\Users\quantbase\Desktop\sydata")
DATA_ROOT = Path(r"C:\Users\quantbase\Desktop\marketdata")
SRC = REPO_ROOT / "src"

# Make `sydata` importable
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Stabilize relative paths (scripts/, etc.)
os.chdir(str(REPO_ROOT))

print("python:", sys.executable)
print("cwd:", Path.cwd())
print("SRC:", SRC)
print("SRC exists:", SRC.exists())

# Hard fail early if repo importability is broken
import sydata  
print("sydata:", sydata.__file__)

python: c:\Users\quantbase\.conda\envs\sydata-311\python.exe
cwd: C:\Users\quantbase\Desktop\sydata
SRC: C:\Users\quantbase\Desktop\sydata\src
SRC exists: True
sydata: C:\Users\quantbase\Desktop\sydata\src\sydata\__init__.py


In [ ]:
# Cell 1 — Run config (single source of truth)

from pathlib import Path  

# --- Run identity (new folder each time to avoid mixing partitions) ---
RUN_ID = "e2e_microagg_20260206"  # change freely

# --- Timeframe + scope ---
INTERVAL = "15m"
START = "2025-01-01"
END_EXCL = "2026-01-01"

# --- Symbol selection ---
BASKET = "core_major"   # uses DATA_ROOT/meta/symbols.yml

# --- Canonical roots ---
NORM_ROOT = DATA_ROOT / "norm"
META_ROOT = DATA_ROOT / "meta"

MANIFEST_PATH = META_ROOT / "symbols.yml"
RAW_AGG_ROOT = DATA_ROOT / "raw" / "binance" / "spot_aggtrades"
MASTER_ROOT = NORM_ROOT / "master"  # baseline master already exists here

# --- Clean, isolated output roots for THIS run ---
RUN_ROOT = NORM_ROOT / "_runs" / RUN_ID

MICRO_AGG_NORM_ROOT = RUN_ROOT / "spot_aggtrades_resampled_micro"
JOINED_MONTH_ROOT   = RUN_ROOT / "master_plus_microaggtrades"
FINAL_LONG_ROOT     = RUN_ROOT / "master_long_plus_microaggtrades"
FINAL_WIDE_ROOT     = RUN_ROOT / "master_wide_plus_microaggtrades"

# Create run dirs (idempotent)
for p in [RUN_ROOT, MICRO_AGG_NORM_ROOT, JOINED_MONTH_ROOT, FINAL_LONG_ROOT, FINAL_WIDE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Quick path audit (do not proceed if any required root is missing)
path_audit = {
    "REPO_ROOT": str(REPO_ROOT),
    "DATA_ROOT": str(DATA_ROOT),
    "SRC": str(SRC),
    "NORM_ROOT": str(NORM_ROOT),
    "META_ROOT": str(META_ROOT),
    "MANIFEST_PATH": str(MANIFEST_PATH),
    "RAW_AGG_ROOT": str(RAW_AGG_ROOT),
    "MASTER_ROOT": str(MASTER_ROOT),
    "RUN_ROOT": str(RUN_ROOT),
    "MICRO_AGG_NORM_ROOT": str(MICRO_AGG_NORM_ROOT),
    "JOINED_MONTH_ROOT": str(JOINED_MONTH_ROOT),
    "FINAL_LONG_ROOT": str(FINAL_LONG_ROOT),
    "FINAL_WIDE_ROOT": str(FINAL_WIDE_ROOT),
}

exists_audit = {
    "MANIFEST_PATH_exists": MANIFEST_PATH.exists(),
    "RAW_AGG_ROOT_exists": RAW_AGG_ROOT.exists(),
    "MASTER_ROOT_exists": MASTER_ROOT.exists(),
}

print(json.dumps(path_audit, indent=2))
print(json.dumps(exists_audit, indent=2))

# Hard fail early on missing canonical inputs
assert exists_audit["MANIFEST_PATH_exists"], f"Missing manifest: {MANIFEST_PATH}"
assert exists_audit["RAW_AGG_ROOT_exists"], f"Missing raw agg root: {RAW_AGG_ROOT}"
assert exists_audit["MASTER_ROOT_exists"], f"Missing master root: {MASTER_ROOT}"

{
  "REPO_ROOT": "C:\\Users\\quantbase\\Desktop\\sydata",
  "DATA_ROOT": "C:\\Users\\quantbase\\Desktop\\marketdata",
  "SRC": "C:\\Users\\quantbase\\Desktop\\sydata\\src",
  "NORM_ROOT": "C:\\Users\\quantbase\\Desktop\\marketdata\\norm",
  "META_ROOT": "C:\\Users\\quantbase\\Desktop\\marketdata\\meta",
  "MANIFEST_PATH": "C:\\Users\\quantbase\\Desktop\\marketdata\\meta\\symbols.yml",
  "RAW_AGG_ROOT": "C:\\Users\\quantbase\\Desktop\\marketdata\\raw\\binance\\spot_aggtrades",
  "MASTER_ROOT": "C:\\Users\\quantbase\\Desktop\\marketdata\\norm\\master",
  "RUN_ROOT": "C:\\Users\\quantbase\\Desktop\\marketdata\\norm\\_runs\\e2e_microagg_20260206",
  "MICRO_AGG_NORM_ROOT": "C:\\Users\\quantbase\\Desktop\\marketdata\\norm\\_runs\\e2e_microagg_20260206\\spot_aggtrades_resampled_micro",
  "JOINED_MONTH_ROOT": "C:\\Users\\quantbase\\Desktop\\marketdata\\norm\\_runs\\e2e_microagg_20260206\\master_plus_microaggtrades",
  "FINAL_LONG_ROOT": "C:\\Users\\quantbase\\Desktop\\marketdata\\norm\\_runs\\

In [ ]:
from __future__ import annotations  

from pathlib import Path  
import json  

import pandas as pd  
import yaml  


# --- run config (use what you already printed in cells 0/1) ---
BASKET = "core_major"
INTERVAL = "15m"

START = "2025-01-01"      # inclusive
END_EXCL = "2026-01-01"   # exclusive


def interval_to_pandas_freq(interval: str) -> str:
    """
    '15m' -> '15min', '1h' -> '1H', '1d' -> '1D'
    (dt.floor / date_range friendly)
    """
    interval = interval.strip().lower()
    if interval.endswith("m"):
        return f"{int(interval[:-1])}min"
    if interval.endswith("h"):
        return f"{int(interval[:-1])}H"
    if interval.endswith("d"):
        return f"{int(interval[:-1])}D"
    return interval


def ensure_utc(ts: pd.Timestamp) -> pd.Timestamp:
    t = pd.Timestamp(ts)
    if t.tzinfo is None:
        return t.tz_localize("UTC")
    return t.tz_convert("UTC")


def iter_year_months(start_utc: pd.Timestamp, end_excl_utc: pd.Timestamp) -> list[tuple[int, int]]:
    """
    Months intersecting [start, end_excl) without using Period (avoids tz-drop warnings).
    """
    s = ensure_utc(start_utc)
    e = ensure_utc(end_excl_utc) - pd.Timedelta(microseconds=1)  # inclusive end moment
    months: list[tuple[int, int]] = []
    y, m = s.year, s.month
    y2, m2 = e.year, e.month

    while (y < y2) or (y == y2 and m <= m2):
        months.append((y, m))
        m += 1
        if m == 13:
            m = 1
            y += 1
    return months


def load_symbols_from_manifest(manifest_path: Path, basket: str) -> list[str]:
    spec = yaml.safe_load(manifest_path.read_text(encoding="utf-8"))
    if not isinstance(spec, dict):
        raise ValueError("Manifest did not parse to a dict")

    baskets = spec.get("baskets", {})
    if not isinstance(baskets, dict) or basket not in baskets:
        raise ValueError(f"Basket '{basket}' not found in manifest")

    node = baskets[basket]
    # supports:
    # baskets: { core_major: { symbols: [...] } }
    # baskets: { core_major: [...] }
    if isinstance(node, dict):
        symbols = node.get("symbols")
    else:
        symbols = node

    if not isinstance(symbols, list) or not symbols or not all(isinstance(x, str) for x in symbols):
        raise ValueError(f"Basket '{basket}' must be list[str] under baskets.{basket}.symbols (or direct list)")

    # drop dups, preserve order
    return list(dict.fromkeys(symbols))


START_UTC = ensure_utc(pd.Timestamp(START))
END_EXCL_UTC = ensure_utc(pd.Timestamp(END_EXCL))
FREQ = interval_to_pandas_freq(INTERVAL)

SYMBOLS = load_symbols_from_manifest(MANIFEST_PATH, BASKET)
YEAR_MONTHS = iter_year_months(START_UTC, END_EXCL_UTC)

print(json.dumps({
    "BASKET": BASKET,
    "INTERVAL": INTERVAL,
    "FREQ": FREQ,
    "START_UTC": str(START_UTC),
    "END_EXCL_UTC": str(END_EXCL_UTC),
    "symbols_n": len(SYMBOLS),
    "symbols": SYMBOLS,
    "year_months_n": len(YEAR_MONTHS),
    "year_months_head": YEAR_MONTHS[:5],
    "year_months_tail": YEAR_MONTHS[-5:],
}, indent=2))

{
  "BASKET": "core_major",
  "INTERVAL": "15m",
  "FREQ": "15min",
  "START_UTC": "2025-01-01 00:00:00+00:00",
  "END_EXCL_UTC": "2026-01-01 00:00:00+00:00",
  "symbols_n": 7,
  "symbols": [
    "BTC-USDT",
    "ETH-USDT",
    "SOL-USDT",
    "BNB-USDT",
    "XRP-USDT",
    "ADA-USDT",
    "LINK-USDT"
  ],
  "year_months_n": 12,
  "year_months_head": [
    [
      2025,
      1
    ],
    [
      2025,
      2
    ],
    [
      2025,
      3
    ],
    [
      2025,
      4
    ],
    [
      2025,
      5
    ]
  ],
  "year_months_tail": [
    [
      2025,
      8
    ],
    [
      2025,
      9
    ],
    [
      2025,
      10
    ],
    [
      2025,
      11
    ],
    [
      2025,
      12
    ]
  ]
}


In [ ]:
import pandas as pd  
import pyarrow.parquet as pq  


RAW_REQUIRED = ["ts", "agg_trade_id", "price", "qty", "is_buyer_maker", "symbol"]
RAW_OPTIONAL = ["venue", "dataset"]


def raw_month_dir(raw_root: Path, symbol: str, year: int, month: int) -> Path:
    return raw_root / f"symbol={symbol}" / f"year={year}" / f"month={month:02d}"


def list_parquet_files(p: Path) -> list[Path]:
    if not p.exists():
        return []
    return sorted(p.glob("part-*.parquet"))


# --- coverage table: do we have raw files for every (symbol, month)? ---
rows = []
for sym in SYMBOLS:
    for (y, m) in YEAR_MONTHS:
        d = raw_month_dir(RAW_AGG_ROOT, sym, y, m)
        files = list_parquet_files(d)
        rows.append({
            "symbol": sym,
            "year": y,
            "month": m,
            "dir_exists": d.exists(),
            "n_files": len(files),
            "example_file": str(files[0]) if files else None,
        })

raw_cov = pd.DataFrame(rows).sort_values(["symbol", "year", "month"]).reset_index(drop=True)

missing_dirs = raw_cov.loc[~raw_cov["dir_exists"]]
empty_months = raw_cov.loc[(raw_cov["dir_exists"]) & (raw_cov["n_files"] == 0)]

print("RAW coverage summary:")
print({
    "checks": int(len(raw_cov)),
    "missing_dirs": int(len(missing_dirs)),
    "empty_months": int(len(empty_months)),
    "ok_months": int((raw_cov["dir_exists"] & (raw_cov["n_files"] > 0)).sum()),
})

# show only failures (if any)
if len(missing_dirs):
    display(missing_dirs.head(20))
if len(empty_months):
    display(empty_months.head(20))

# --- schema smoke: read only first row-group from a representative file per symbol ---
# pick a mid-point month in the run (avoids hardcoding month=07)
mid_ym = YEAR_MONTHS[len(YEAR_MONTHS) // 2]
SAMPLE_YEAR, SAMPLE_MONTH = mid_ym

schema_rows = []
for sym in SYMBOLS:
    d = raw_month_dir(RAW_AGG_ROOT, sym, SAMPLE_YEAR, SAMPLE_MONTH)
    files = list_parquet_files(d)
    if not files:
        schema_rows.append({
            "symbol": sym, "year": SAMPLE_YEAR, "month": SAMPLE_MONTH,
            "ok": False, "error": "no parquet files in sample month",
        })
        continue

    f0 = files[0]
    try:
        pf = pq.ParquetFile(f0)
        cols = RAW_REQUIRED + [c for c in RAW_OPTIONAL if c in pf.schema.names]
        df0 = pf.read_row_group(0, columns=cols).to_pandas()

        # light checks
        missing_cols = [c for c in RAW_REQUIRED if c not in df0.columns]
        ts_dtype = str(df0["ts"].dtype) if "ts" in df0.columns else None
        sym_ok = True
        if "symbol" in df0.columns and len(df0) > 0:
            sym_ok = bool((df0["symbol"] == sym).all())

        schema_rows.append({
            "symbol": sym,
            "year": SAMPLE_YEAR,
            "month": SAMPLE_MONTH,
            "ok": (len(missing_cols) == 0) and sym_ok,
            "file": str(f0),
            "missing_cols": missing_cols,
            "ts_dtype": ts_dtype,
            "rowgroup0_rows": int(len(df0)),
            "symbol_matches_folder": sym_ok,
            "agg_trade_id_dtype": str(df0["agg_trade_id"].dtype) if "agg_trade_id" in df0.columns else None,
            "is_buyer_maker_dtype": str(df0["is_buyer_maker"].dtype) if "is_buyer_maker" in df0.columns else None,
        })
    except Exception as e:
        schema_rows.append({
            "symbol": sym, "year": SAMPLE_YEAR, "month": SAMPLE_MONTH,
            "ok": False, "file": str(f0), "error": repr(e),
        })

raw_schema_smoke = pd.DataFrame(schema_rows).sort_values(["ok", "symbol"], ascending=[True, True]).reset_index(drop=True)
print("RAW schema smoke summary:", {
    "checks": int(len(raw_schema_smoke)),
    "passed": int(raw_schema_smoke["ok"].sum()),
    "failed": int((~raw_schema_smoke["ok"]).sum()),
    "sample_year": SAMPLE_YEAR,
    "sample_month": SAMPLE_MONTH,
})

display(raw_schema_smoke)



RAW coverage summary:
{'checks': 84, 'missing_dirs': 0, 'empty_months': 0, 'ok_months': 84}
RAW schema smoke summary: {'checks': 7, 'passed': 7, 'failed': 0, 'sample_year': 2025, 'sample_month': 7}


,symbol,year,month,ok,file,missing_cols,ts_dtype,rowgroup0_rows,symbol_matches_folder,agg_trade_id_dtype,is_buyer_maker_dtype
0,ADA-USDT,2025,7,True,C:\Users\quantbase\Desktop\marketdata\raw\bina...,[],"datetime64[ns, UTC]",1048576,True,Int64,boolean
1,BNB-USDT,2025,7,True,C:\Users\quantbase\Desktop\marketdata\raw\bina...,[],"datetime64[ns, UTC]",1048576,True,Int64,boolean
2,BTC-USDT,2025,7,True,C:\Users\quantbase\Desktop\marketdata\raw\bina...,[],"datetime64[ns, UTC]",1048576,True,Int64,boolean
3,ETH-USDT,2025,7,True,C:\Users\quantbase\Desktop\marketdata\raw\bina...,[],"datetime64[ns, UTC]",1048576,True,Int64,boolean
4,LINK-USDT,2025,7,True,C:\Users\quantbase\Desktop\marketdata\raw\bina...,[],"datetime64[ns, UTC]",1048576,True,Int64,boolean
5,SOL-USDT,2025,7,True,C:\Users\quantbase\Desktop\marketdata\raw\bina...,[],"datetime64[ns, UTC]",1048576,True,Int64,boolean
6,XRP-USDT,2025,7,True,C:\Users\quantbase\Desktop\marketdata\raw\bina...,[],"datetime64[ns, UTC]",1048576,True,Int64,boolean


In [7]:
# If you want a quick peek at the two largest symbols’ rowgroup0:
#display(raw_schema_smoke.loc[raw_schema_smoke["symbol"].isin(["BTC-USDT","ADA-USDT"])])

In [ ]:
# (SANITY_CHECK) raw aggtrades

import pyarrow.dataset as ds  
import pyarrow as pa  


# --- choose a representative day inside the run window (UTC) ---
SAMPLE_DAY = pd.Timestamp("2025-07-01", tz="UTC")
DAY_START = SAMPLE_DAY
DAY_END_EXCL = SAMPLE_DAY + pd.Timedelta(days=1)

# columns we validate at raw level
RAW_COLS = ["ts", "agg_trade_id", "price", "qty", "is_buyer_maker", "symbol"]


def raw_month_dir(raw_root: Path, symbol: str, year: int, month: int) -> Path:
    return raw_root / f"symbol={symbol}" / f"year={year}" / f"month={month:02d}"


def list_parquet_files(p: Path) -> list[Path]:
    if not p.exists():
        return []
    return sorted(p.glob("part-*.parquet"))


def read_raw_day(files: list[Path], day_start: pd.Timestamp, day_end_excl: pd.Timestamp) -> pd.DataFrame:
    """
    Reads only the [day_start, day_end_excl) slice across multiple parquet parts.
    Uses pyarrow dataset pushdown filters to avoid loading the full month.
    """
    if not files:
        return pd.DataFrame(columns=RAW_COLS)

    dset = ds.dataset([str(f) for f in files], format="parquet")
    # Arrow scalar timestamps (keep tz-awareness from pandas Timestamp string form)
    filt = (ds.field("ts") >= pa.scalar(day_start.to_pydatetime())) & (ds.field("ts") < pa.scalar(day_end_excl.to_pydatetime()))
    tbl = dset.to_table(columns=RAW_COLS, filter=filt)
    df = tbl.to_pandas()
    # ensure pandas ts is UTC-aware (defensive)
    if "ts" in df.columns and not pd.api.types.is_datetime64_any_dtype(df["ts"]):
        df["ts"] = pd.to_datetime(df["ts"], utc=True)
    else:
        # if already datetime but tz-naive, localize
        if "ts" in df.columns and getattr(df["ts"].dt, "tz", None) is None:
            df["ts"] = df["ts"].dt.tz_localize("UTC")
    return df


def qc_raw_day(df: pd.DataFrame, symbol: str, day_start: pd.Timestamp, day_end_excl: pd.Timestamp) -> dict:
    out: dict = {
        "ok": False,
        "symbol": symbol,
        "day": str(day_start),
        "rows": int(len(df)),
    }
    if len(df) == 0:
        out["error"] = "no rows for day"
        return out

    missing = [c for c in RAW_COLS if c not in df.columns]
    if missing:
        out["error"] = f"missing_cols={missing}"
        return out

    # basic validity
    out["ts_min"] = str(df["ts"].min())
    out["ts_max"] = str(df["ts"].max())
    out["in_day_bounds"] = bool((df["ts"].min() >= day_start) and (df["ts"].max() < day_end_excl))

    out["price_nonpos"] = int((df["price"] <= 0).sum())
    out["qty_nonpos"] = int((df["qty"] <= 0).sum())
    out["is_buyer_maker_na"] = int(df["is_buyer_maker"].isna().sum())
    out["symbol_mismatch_rows"] = int((df["symbol"] != symbol).sum())

    # ordering checks (Binance aggTradeId should be strictly increasing over time; allow same-ms ts jitter)
    df2 = df.sort_values(["ts", "agg_trade_id"]).reset_index(drop=True)
    diffs = df2["agg_trade_id"].astype("Int64").diff()

    out["trade_id_noninc"] = int((diffs <= 0).fillna(False).sum())  # <=0 catches duplicates and reversals
    out["trade_id_dups"] = int(df2["agg_trade_id"].duplicated().sum())

    # maker/taker balance sanity (should have both sides most of the time; not a hard gate)
    taker_buy = (df2["is_buyer_maker"] == False).sum()  # noqa: E712
    taker_sell = (df2["is_buyer_maker"] == True).sum()  # noqa: E712
    out["taker_buy_rows"] = int(taker_buy)
    out["taker_sell_rows"] = int(taker_sell)
    out["taker_buy_frac"] = float(taker_buy / len(df2))
    out["taker_sell_frac"] = float(taker_sell / len(df2))

    # final ok gate
    out["ok"] = (
        out["in_day_bounds"]
        and out["price_nonpos"] == 0
        and out["qty_nonpos"] == 0
        and out["is_buyer_maker_na"] == 0
        and out["symbol_mismatch_rows"] == 0
        and out["trade_id_noninc"] == 0
        and out["trade_id_dups"] == 0
    )
    return out


# --- run QC for all symbols on the sample day ---
rows = []
for sym in SYMBOLS:
    month_dir = raw_month_dir(RAW_AGG_ROOT, sym, SAMPLE_DAY.year, SAMPLE_DAY.month)
    files = list_parquet_files(month_dir)
    df_day = read_raw_day(files, DAY_START, DAY_END_EXCL)
    rows.append(qc_raw_day(df_day, sym, DAY_START, DAY_END_EXCL))

raw_day_qc = pd.DataFrame(rows).sort_values(["ok", "symbol"], ascending=[True, True]).reset_index(drop=True)

summary = {
    "checks": int(len(raw_day_qc)),
    "passed": int(raw_day_qc["ok"].sum()),
    "failed": int((~raw_day_qc["ok"]).sum()),
    "day": str(DAY_START),
}
print("RAW day QC summary:", json.dumps(summary, indent=2))
display(raw_day_qc)

RAW day QC summary: {
  "checks": 7,
  "passed": 7,
  "failed": 0,
  "day": "2025-07-01 00:00:00+00:00"
}


,ok,symbol,day,rows,ts_min,ts_max,in_day_bounds,price_nonpos,qty_nonpos,is_buyer_maker_na,symbol_mismatch_rows,trade_id_noninc,trade_id_dups,taker_buy_rows,taker_sell_rows,taker_buy_frac,taker_sell_frac
0,True,ADA-USDT,2025-07-01 00:00:00+00:00,85141,2025-07-01 00:00:03.812295+00:00,2025-07-01 23:59:57.330741+00:00,True,0,0,0,0,0,0,43898,41243,0.515592,0.484408
1,True,BNB-USDT,2025-07-01 00:00:00+00:00,149313,2025-07-01 00:00:00.185834+00:00,2025-07-01 23:59:59.540261+00:00,True,0,0,0,0,0,0,84393,64920,0.565209,0.434791
2,True,BTC-USDT,2025-07-01 00:00:00+00:00,802159,2025-07-01 00:00:00.015562+00:00,2025-07-01 23:59:58.779518+00:00,True,0,0,0,0,0,0,386914,415245,0.482341,0.517659
3,True,ETH-USDT,2025-07-01 00:00:00+00:00,670718,2025-07-01 00:00:00.345232+00:00,2025-07-01 23:59:58.696041+00:00,True,0,0,0,0,0,0,328656,342062,0.490006,0.509994
4,True,LINK-USDT,2025-07-01 00:00:00+00:00,21949,2025-07-01 00:00:27.310453+00:00,2025-07-01 23:59:49.226251+00:00,True,0,0,0,0,0,0,11025,10924,0.502301,0.497699
5,True,SOL-USDT,2025-07-01 00:00:00+00:00,247535,2025-07-01 00:00:00.049881+00:00,2025-07-01 23:59:57.815303+00:00,True,0,0,0,0,0,0,134095,113440,0.541721,0.458279
6,True,XRP-USDT,2025-07-01 00:00:00+00:00,224466,2025-07-01 00:00:00.143327+00:00,2025-07-01 23:59:57.333774+00:00,True,0,0,0,0,0,0,102959,121507,0.458684,0.541316


In [ ]:
# Import from repo

from __future__ import annotations  

import inspect  
import importlib  
import json  

import pandas as pd  
import numpy as np  

import sydata  
from sydata.io import symbols as sym_io  

# datasets we care about for this notebook stage
from sydata.datasets import spot_aggtrades_resampled as agg_mod  
from sydata.datasets import master_join_aggtrades as join_mod  


def _sig(obj) -> str:
    try:
        return str(inspect.signature(obj))
    except Exception:
        return "<no signature>"


def _describe_module(mod, names: list[str]) -> dict:
    info = {"module": mod.__name__, "file": getattr(mod, "__file__", None), "items": {}}
    print(f"\n{mod.__name__}  ->  {info['file']}")
    for n in names:
        obj = getattr(mod, n, None)
        ok = obj is not None
        info["items"][n] = {"present": ok, "sig": _sig(obj) if ok else None}
        if ok:
            print(f"  OK  {n}: {_sig(obj)}")
        else:
            print(f"  MISSING  {n}")
    return info


# what we expect to exist somewhere (this is a *checklist*, not an assumption)
sym_names = ["load_manifest", "load_basket"]
agg_names = [
    # cfg / plumbing
    "AggResampleCfg",
    "iter_year_months",
    "resolve_symbols",
    "resampled_out_path",
    # core ops
    "resample_month",
    "load_resampled_month",
]
join_names = [
    # cfg / plumbing
    "MasterAggJoinCfg",
    "iter_year_months",
    "resolve_symbols",
    # core ops
    "join_master_with_aggtrades",
    "run_monthly_join",
]

report = {
    "sydata": {"file": sydata.__file__},
    "symbols_io": _describe_module(sym_io, sym_names),
    "agg_resample": _describe_module(agg_mod, agg_names),
    "master_join": _describe_module(join_mod, join_names),
}

# quick missing summary (so failures are explicit and local)
missing = {}
for k, v in report.items():
    if isinstance(v, dict) and "items" in v:
        miss = [name for name, meta in v["items"].items() if not meta["present"]]
        if miss:
            missing[k] = miss

print("\nMISSING SUMMARY:", json.dumps(missing, indent=2))

# keep this object for later cells
API_REPORT = report
API_REPORT


sydata.io.symbols  ->  C:\Users\quantbase\Desktop\sydata\src\sydata\io\symbols.py
  OK  load_manifest: (path: 'Path') -> 'dict[str, Any]'
  OK  load_basket: (spec: 'dict[str, Any]', basket_name: 'str') -> 'list[str]'

sydata.datasets.spot_aggtrades_resampled  ->  C:\Users\quantbase\Desktop\sydata\src\sydata\datasets\spot_aggtrades_resampled.py
  OK  AggResampleCfg: (data_root: 'Path', manifest_path: 'Path', basket: 'str', interval: 'str', start: 'str', end_excl: 'str', symbols_override: 'list[str] | None' = None, raw_root: 'Path | None' = None, norm_root: 'Path | None' = None) -> None
  OK  iter_year_months: (start_utc: 'pd.Timestamp', end_excl_utc: 'pd.Timestamp') -> 'list[tuple[int, int]]'
  OK  resolve_symbols: (cfg: 'AggResampleCfg') -> 'list[str]'
  OK  resampled_out_path: (norm_root: 'Path', *, interval: 'str', symbol: 'str', year: 'int', month: 'int') -> 'Path'
  OK  resample_month: (cfg: 'AggResampleCfg', symbol: 'str', year: 'int', month: 'int') -> 'dict'
  OK  load_resampled

{'sydata': {'file': 'C:\\Users\\quantbase\\Desktop\\sydata\\src\\sydata\\__init__.py'},
 'symbols_io': {'module': 'sydata.io.symbols',
  'file': 'C:\\Users\\quantbase\\Desktop\\sydata\\src\\sydata\\io\\symbols.py',
  'items': {'load_manifest': {'present': True,
    'sig': "(path: 'Path') -> 'dict[str, Any]'"},
   'load_basket': {'present': True,
    'sig': "(spec: 'dict[str, Any]', basket_name: 'str') -> 'list[str]'"}}},
 'agg_resample': {'module': 'sydata.datasets.spot_aggtrades_resampled',
  'file': 'C:\\Users\\quantbase\\Desktop\\sydata\\src\\sydata\\datasets\\spot_aggtrades_resampled.py',
  'items': {'AggResampleCfg': {'present': True,
    'sig': "(data_root: 'Path', manifest_path: 'Path', basket: 'str', interval: 'str', start: 'str', end_excl: 'str', symbols_override: 'list[str] | None' = None, raw_root: 'Path | None' = None, norm_root: 'Path | None' = None) -> None"},
   'iter_year_months': {'present': True,
    'sig': "(start_utc: 'pd.Timestamp', end_excl_utc: 'pd.Timestamp') ->

In [ ]:
# (RESAMPLE) (TEST) 1 month 


from sydata.datasets.spot_aggtrades_resampled import (  
    AggResampleCfg,
    resample_month,
    load_resampled_month,
    resampled_out_path,
)
from sydata.datasets import master_join_aggtrades as join_mod  

# ---- smoke params (keep small, target the known troublesome month) ----
SMOKE_YEAR = 2025
SMOKE_MONTH = 7
SMOKE_SYMBOLS = ["ADA-USDT", "BTC-USDT"]

# ---- build paths (using your notebook globals from earlier cells) ----
# expects these already defined in earlier cells:
# DATA_ROOT, RAW_AGG_ROOT, MASTER_ROOT, MANIFEST_PATH,
# MICRO_AGG_NORM_ROOT, JOINED_MONTH_ROOT, INTERVAL
assert MANIFEST_PATH.exists(), MANIFEST_PATH
assert RAW_AGG_ROOT.exists(), RAW_AGG_ROOT
assert MASTER_ROOT.exists(), MASTER_ROOT
Path(MICRO_AGG_NORM_ROOT).mkdir(parents=True, exist_ok=True)
Path(JOINED_MONTH_ROOT).mkdir(parents=True, exist_ok=True)

# ---- helpers ----
def master_month_path(master_root: Path, *, interval: str, symbol: str, year: int, month: int) -> Path:
    return (
        master_root
        / f"interval={interval}"
        / f"year={year}"
        / f"month={month:02d}"
        / f"symbol={symbol}"
        / f"part-{year}-{month:02d}.parquet"
    )

def joined_month_path(joined_root: Path, *, interval: str, symbol: str, year: int, month: int) -> Path:
    return (
        joined_root
        / f"interval={interval}"
        / f"year={year}"
        / f"month={month:02d}"
        / f"symbol={symbol}"
        / f"part-{year}-{month:02d}.parquet"
    )

def qc_micro_bucket(df: pd.DataFrame, eps_qty: float = 1e-6, eps_notional: float = 1e-4) -> dict:
    # Required columns for micro agg
    req = [
        "ts",
        "sum_qty","trades","cvd_qty","vwap","last_trade_id",
        "taker_buy_qty","taker_sell_qty",
        "taker_buy_trades","taker_sell_trades",
        "buy_notional","sell_notional",
        "buy_vwap","sell_vwap",
        "first_trade_id","first_price",
        "last_price",
        "symbol",
    ]
    missing = [c for c in req if c not in df.columns]
    out = {"ok": False, "missing_cols": missing}
    if missing:
        return out

    d = df.sort_values("ts").reset_index(drop=True)

    # structural
    out["rows"] = int(len(d))
    out["ts_unique"] = bool(d["ts"].is_unique)
    out["ts_monotonic"] = bool(d["ts"].is_monotonic_increasing)

    # mass / count balances
    out["mass_max_abs"] = float((d["taker_buy_qty"] + d["taker_sell_qty"] - d["sum_qty"]).abs().max())
    out["count_max_abs"] = float((d["taker_buy_trades"] + d["taker_sell_trades"] - d["trades"]).abs().max())

    # vwap identity in quote terms (allow tiny fp error)
    # agg_vwap * sum_qty should equal buy_notional + sell_notional
    lhs = (d["buy_notional"] + d["sell_notional"])
    rhs = (d["vwap"] * d["sum_qty"])
    out["vwap_id_max_abs"] = float((lhs - rhs).abs().max())

    # cvd consistency
    out["cvd_max_abs"] = float((d["cvd_qty"] - (d["taker_buy_qty"] - d["taker_sell_qty"])).abs().max())

    # first/last ordering
    out["first_le_last_trade_id_all"] = bool((d["first_trade_id"] <= d["last_trade_id"]).all())
    out["last_trade_id_monotone"] = bool(d["last_trade_id"].is_monotonic_increasing)

    # NaN hygiene for critical price/vwap fields
    for c in ["vwap","buy_vwap","sell_vwap","first_price","last_price"]:
        out[f"{c}_nan_frac"] = float(pd.isna(d[c]).mean())
        out[f"{c}_inf_frac"] = float(np.isinf(d[c].astype(float)).mean())  # safe cast (these are numeric columns)

    out["ok"] = (
        out["ts_unique"]
        and out["ts_monotonic"]
        and out["mass_max_abs"] <= eps_qty
        and out["count_max_abs"] == 0.0
        and out["vwap_id_max_abs"] <= eps_notional
        and out["cvd_max_abs"] <= eps_qty
        and out["first_le_last_trade_id_all"]
        and out["last_trade_id_monotone"]
        and all(out[f"{c}_nan_frac"] == 0.0 and out[f"{c}_inf_frac"] == 0.0 for c in ["vwap","first_price","last_price"])
    )
    return out


# ---- 1) micro-resample smoke (writes into MICRO_AGG_NORM_ROOT) ----
RES_CFG = AggResampleCfg(
    data_root=DATA_ROOT,
    manifest_path=MANIFEST_PATH,
    basket=BASKET,
    interval=INTERVAL,
    start=START_UTC.strftime("%Y-%m-%d"),
    end_excl=END_EXCL_UTC.strftime("%Y-%m-%d"),
    raw_root=RAW_AGG_ROOT,
    norm_root=Path(MICRO_AGG_NORM_ROOT),
)

micro_reports = {}

for sym in SMOKE_SYMBOLS:
    # run resample for the month
    res_info = resample_month(RES_CFG, sym, SMOKE_YEAR, SMOKE_MONTH)

    out_p = resampled_out_path(
        Path(MICRO_AGG_NORM_ROOT),
        interval=INTERVAL,
        symbol=sym,
        year=SMOKE_YEAR,
        month=SMOKE_MONTH,
    )
    exists = out_p.exists()

    # load + qc
    df_micro = load_resampled_month(RES_CFG, sym, SMOKE_YEAR, SMOKE_MONTH)
    qc = qc_micro_bucket(df_micro)

    micro_reports[sym] = {
        "resample_info": res_info,
        "out_path": str(out_p),
        "out_exists": bool(exists),
        "micro_shape": tuple(df_micro.shape),
        "qc": qc,
    }

micro_reports

{'ADA-USDT': {'resample_info': {'ok': True,
   'symbol': 'ADA-USDT',
   'year': 2025,
   'month': 7,
   'out': 'C:\\Users\\quantbase\\Desktop\\marketdata\\norm\\_runs\\e2e_microagg_20260206\\spot_aggtrades_resampled_micro\\interval=15m\\year=2025\\month=07\\symbol=ADA-USDT\\part-2025-07.parquet',
   'rows': 2976,
   'min_ts': '2025-07-01T00:00:00+00:00',
   'max_ts': '2025-07-31T23:45:00+00:00'},
  'out_path': 'C:\\Users\\quantbase\\Desktop\\marketdata\\norm\\_runs\\e2e_microagg_20260206\\spot_aggtrades_resampled_micro\\interval=15m\\year=2025\\month=07\\symbol=ADA-USDT\\part-2025-07.parquet',
  'out_exists': True,
  'micro_shape': (2976, 18),
  'qc': {'ok': True,
   'missing_cols': [],
   'rows': 2976,
   'ts_unique': True,
   'ts_monotonic': True,
   'mass_max_abs': 3.725290298461914e-09,
   'count_max_abs': 0.0,
   'vwap_id_max_abs': 9.313225746154785e-10,
   'cvd_max_abs': 0.0,
   'first_le_last_trade_id_all': True,
   'last_trade_id_monotone': True,
   'vwap_nan_frac': 0.0,
   'vw

In [ ]:
# (JOIN) (TEST) 1 month

from pathlib import Path  
import pandas as pd  

join_reports = {}

MICRO_PREFIX_COLS = [
    "taker_buy_qty","taker_sell_qty",
    "taker_buy_trades","taker_sell_trades",
    "buy_notional","sell_notional",
    "buy_vwap","sell_vwap",
    "first_price","last_price",
    "first_trade_id",
]

for sym in SMOKE_SYMBOLS:
    master_p = master_month_path(Path(MASTER_ROOT), interval=INTERVAL, symbol=sym, year=SMOKE_YEAR, month=SMOKE_MONTH)
    assert master_p.exists(), master_p

    # master month is per-symbol; it often lacks `symbol` column, but the join code expects symbol alignment
    md = pd.read_parquet(master_p)
    if "symbol" not in md.columns:
        md["symbol"] = sym

    ad = load_resampled_month(RES_CFG, sym, SMOKE_YEAR, SMOKE_MONTH)

    # ensure micro columns exist before join (prevents silent partial joins later)
    missing_micro = [c for c in MICRO_PREFIX_COLS if c not in ad.columns]
    assert len(missing_micro) == 0, (sym, missing_micro)

    jd = join_mod.join_master_with_aggtrades(md, ad)

    # write joined month into run sandbox
    out_p = joined_month_path(Path(JOINED_MONTH_ROOT), interval=INTERVAL, symbol=sym, year=SMOKE_YEAR, month=SMOKE_MONTH)
    out_p.parent.mkdir(parents=True, exist_ok=True)
    jd.to_parquet(out_p, index=False)

    # critical check: non-BTC must not be all-NaN for micro columns
    micro_joined_cols = [f"agg_{c}" for c in MICRO_PREFIX_COLS]
    present = [c for c in micro_joined_cols if c in jd.columns]

    nan_fracs = {c: float(jd[c].isna().mean()) for c in present}

    join_reports[sym] = {
        "master_path": str(master_p),
        "joined_out": str(out_p),
        "rows": int(len(jd)),
        "dup_ts_symbol": int(jd.duplicated(["ts","symbol"]).sum()) if ("ts" in jd.columns and "symbol" in jd.columns) else None,
        "micro_cols_present": present,
        "micro_nan_fracs": nan_fracs,
        "micro_all_nan": {c: (nan_fracs[c] == 1.0) for c in nan_fracs},
    }

join_reports

{'ADA-USDT': {'master_path': 'C:\\Users\\quantbase\\Desktop\\marketdata\\norm\\master\\interval=15m\\year=2025\\month=07\\symbol=ADA-USDT\\part-2025-07.parquet',
  'joined_out': 'C:\\Users\\quantbase\\Desktop\\marketdata\\norm\\_runs\\e2e_microagg_20260206\\master_plus_microaggtrades\\interval=15m\\year=2025\\month=07\\symbol=ADA-USDT\\part-2025-07.parquet',
  'rows': 2976,
  'dup_ts_symbol': 0,
  'micro_cols_present': ['agg_taker_buy_qty',
   'agg_taker_sell_qty',
   'agg_taker_buy_trades',
   'agg_taker_sell_trades',
   'agg_buy_notional',
   'agg_sell_notional',
   'agg_buy_vwap',
   'agg_sell_vwap',
   'agg_first_price',
   'agg_last_price',
   'agg_first_trade_id'],
  'micro_nan_fracs': {'agg_taker_buy_qty': 0.0,
   'agg_taker_sell_qty': 0.0,
   'agg_taker_buy_trades': 0.0,
   'agg_taker_sell_trades': 0.0,
   'agg_buy_notional': 0.0,
   'agg_sell_notional': 0.0,
   'agg_buy_vwap': 0.0,
   'agg_sell_vwap': 0.0,
   'agg_first_price': 0.0,
   'agg_last_price': 0.0,
   'agg_first_trad

In [12]:
# (RESAMPLE) (CANON) ALL symbol-months into MICRO_AGG_NORM_ROOT
# Assumes the following already exist in your notebook from earlier cells:

# SYMBOLS, YEAR_MONTHS, INTERVAL, RES_CFG
# MICRO_AGG_NORM_ROOT (string or Path)
# imports: resample_month, resampled_out_path, load_resampled_month
# qc_micro_bucket(df)

# Safety: ensure this run uses the isolated micro agg root
assert str(Path(RES_CFG.norm_root)).lower() == str(Path(MICRO_AGG_NORM_ROOT)).lower(), (RES_CFG.norm_root, MICRO_AGG_NORM_ROOT)

micro_all = {}  # (symbol, year, month) -> dict report

for sym in SYMBOLS:
    for (y, m) in YEAR_MONTHS:
        # 1) resample (writes parquet)
        res_info = resample_month(RES_CFG, sym, y, m)

        out_p = resampled_out_path(
            Path(MICRO_AGG_NORM_ROOT),
            interval=INTERVAL,
            symbol=sym,
            year=y,
            month=m,
        )

        # 2) load + qc (fast failure: catches wrong schema immediately)
        df_micro = load_resampled_month(RES_CFG, sym, y, m)
        qc = qc_micro_bucket(df_micro)

        micro_all[(sym, y, m)] = {
            "ok": bool(res_info.get("ok", False)) and bool(out_p.exists()) and bool(qc.get("ok", False)),
            "symbol": sym, "year": y, "month": m,
            "out": str(out_p),
            "out_exists": bool(out_p.exists()),
            "rows": int(df_micro.shape[0]),
            "cols": int(df_micro.shape[1]),
            "min_ts": str(df_micro["ts"].min()) if "ts" in df_micro.columns else None,
            "max_ts": str(df_micro["ts"].max()) if "ts" in df_micro.columns else None,
            "qc": qc,
        }

micro_df = (
    pd.DataFrame(list(micro_all.values()))
    .sort_values(["ok", "symbol", "year", "month"], ascending=[True, True, True, True])
    .reset_index(drop=True)
)

micro_summary = {
    "checks": int(len(micro_df)),
    "passed": int(micro_df["ok"].sum()),
    "failed": int((~micro_df["ok"]).sum()),
}

micro_summary, micro_df.head(20)

({'checks': 84, 'passed': 84, 'failed': 0},
       ok    symbol  year  month  \
 0   True  ADA-USDT  2025      1   
 1   True  ADA-USDT  2025      2   
 2   True  ADA-USDT  2025      3   
 3   True  ADA-USDT  2025      4   
 4   True  ADA-USDT  2025      5   
 5   True  ADA-USDT  2025      6   
 6   True  ADA-USDT  2025      7   
 7   True  ADA-USDT  2025      8   
 8   True  ADA-USDT  2025      9   
 9   True  ADA-USDT  2025     10   
 10  True  ADA-USDT  2025     11   
 11  True  ADA-USDT  2025     12   
 12  True  BNB-USDT  2025      1   
 13  True  BNB-USDT  2025      2   
 14  True  BNB-USDT  2025      3   
 15  True  BNB-USDT  2025      4   
 16  True  BNB-USDT  2025      5   
 17  True  BNB-USDT  2025      6   
 18  True  BNB-USDT  2025      7   
 19  True  BNB-USDT  2025      8   
 
                                                   out  out_exists  rows  cols  \
 0   C:\Users\quantbase\Desktop\marketdata\norm\_ru...        True  2976    18   
 1   C:\Users\quantbase\Desktop\ma

In [13]:
# fail check

fails_micro = micro_df.loc[~micro_df["ok"]].copy()
fails_micro[["symbol","year","month","out","out_exists","rows","cols"]].head(50), len(fails_micro)

(Empty DataFrame
 Columns: [symbol, year, month, out, out_exists, rows, cols]
 Index: [],
 0)

In [14]:
# Monthly join ALL symbol-months into JOINED_MONTH_ROOT

#This uses:
# -master month files from MASTER_ROOT
# -micro-resample month files from MICRO_AGG_NORM_ROOT via load_resampled_month
# -join function join_mod.join_master_with_aggtrades(master_df, agg_df)
# -writes joined months under JOINED_MONTH_ROOT

# Micro fields that must not become all-NaN after join
MICRO_PREFIX_COLS = [
    "taker_buy_qty","taker_sell_qty",
    "taker_buy_trades","taker_sell_trades",
    "buy_notional","sell_notional",
    "buy_vwap","sell_vwap",
    "first_price","last_price",
    "first_trade_id",
]
MICRO_JOINED_COLS = [f"agg_{c}" for c in MICRO_PREFIX_COLS]

join_all = {}  # (symbol, year, month) -> dict report

for sym in SYMBOLS:
    for (y, m) in YEAR_MONTHS:
        master_p = master_month_path(Path(MASTER_ROOT), interval=INTERVAL, symbol=sym, year=y, month=m)
        out_p = joined_month_path(Path(JOINED_MONTH_ROOT), interval=INTERVAL, symbol=sym, year=y, month=m)

        rep = {
            "ok": False,
            "symbol": sym, "year": y, "month": m,
            "master": str(master_p),
            "out": str(out_p),
        }

        if not master_p.exists():
            rep["error"] = "missing_master"
            join_all[(sym, y, m)] = rep
            continue

        # Load master and force symbol column (master files are per-symbol)
        md = pd.read_parquet(master_p)
        if "symbol" not in md.columns:
            md["symbol"] = sym

        # Load micro agg from isolated run root
        ad = load_resampled_month(RES_CFG, sym, y, m)

        # Join
        jd = join_mod.join_master_with_aggtrades(md, ad)

        # Basic structural invariants
        rep["rows"] = int(len(jd))
        rep["dup_ts_symbol"] = int(jd.duplicated(["ts","symbol"]).sum())
        rep["joined_has_micro_cols"] = all(c in jd.columns for c in MICRO_JOINED_COLS)

        # NaN fractions for micro columns
        if rep["joined_has_micro_cols"]:
            rep["micro_nan_max"] = float(max(jd[c].isna().mean() for c in MICRO_JOINED_COLS))
            rep["micro_nan_any_all"] = bool(any(jd[c].isna().mean() == 1.0 for c in MICRO_JOINED_COLS))
        else:
            rep["micro_nan_max"] = None
            rep["micro_nan_any_all"] = None

        # Write parquet
        out_p.parent.mkdir(parents=True, exist_ok=True)
        jd.to_parquet(out_p, index=False)
        rep["out_exists"] = bool(out_p.exists())

        rep["ok"] = (
            rep["out_exists"]
            and rep["dup_ts_symbol"] == 0
            and rep["joined_has_micro_cols"]
            and (rep["micro_nan_max"] == 0.0)
            and (rep["micro_nan_any_all"] is False)
        )

        join_all[(sym, y, m)] = rep

join_df = (
    pd.DataFrame(list(join_all.values()))
    .sort_values(["ok","symbol","year","month"], ascending=[True, True, True, True])
    .reset_index(drop=True)
)

join_summary = {
    "checks": int(len(join_df)),
    "passed": int(join_df["ok"].sum()),
    "failed": int((~join_df["ok"]).sum()),
}

join_summary, join_df.head(20)

({'checks': 84, 'passed': 84, 'failed': 0},
       ok    symbol  year  month  \
 0   True  ADA-USDT  2025      1   
 1   True  ADA-USDT  2025      2   
 2   True  ADA-USDT  2025      3   
 3   True  ADA-USDT  2025      4   
 4   True  ADA-USDT  2025      5   
 5   True  ADA-USDT  2025      6   
 6   True  ADA-USDT  2025      7   
 7   True  ADA-USDT  2025      8   
 8   True  ADA-USDT  2025      9   
 9   True  ADA-USDT  2025     10   
 10  True  ADA-USDT  2025     11   
 11  True  ADA-USDT  2025     12   
 12  True  BNB-USDT  2025      1   
 13  True  BNB-USDT  2025      2   
 14  True  BNB-USDT  2025      3   
 15  True  BNB-USDT  2025      4   
 16  True  BNB-USDT  2025      5   
 17  True  BNB-USDT  2025      6   
 18  True  BNB-USDT  2025      7   
 19  True  BNB-USDT  2025      8   
 
                                                master  \
 0   C:\Users\quantbase\Desktop\marketdata\norm\mas...   
 1   C:\Users\quantbase\Desktop\marketdata\norm\mas...   
 2   C:\Users\quantbase\

In [17]:
# show join failures
fails_join = join_df.loc[~join_df["ok"]].copy()
fails_join[["symbol","year","month","master","out","dup_ts_symbol","joined_has_micro_cols","micro_nan_max","micro_nan_any_all"]].head(50), len(fails_join)

(Empty DataFrame
 Columns: [symbol, year, month, master, out, dup_ts_symbol, joined_has_micro_cols, micro_nan_max, micro_nan_any_all]
 Index: [],
 0)

In [18]:
# Define the “final value columns” once (locks wide schema)

BASE_ID_COLS = ["ts", "symbol"]

# Expect these to exist in joined month parts (30 total cols in your pipeline right now)
# ts + symbol + 28 value cols => wide should have 28 * 7 = 196 columns
VALUE_COLS = [
    "open_time",
    "spot_close", "mark_close", "index_close", "premium_close",
    "basis_mark_vs_spot", "basis_index_vs_spot",
    "funding_rate", "funding_interval_hours",
    "volume", "quote_volume", "trades",
    "agg_sum_qty", "agg_trades", "agg_cvd_qty", "agg_vwap", "agg_last_trade_id",
    "agg_taker_buy_qty", "agg_taker_sell_qty",
    "agg_taker_buy_trades", "agg_taker_sell_trades",
    "agg_buy_notional", "agg_sell_notional",
    "agg_buy_vwap", "agg_sell_vwap",
    "agg_first_price", "agg_last_price",
    "agg_first_trade_id",
]

FINAL_LONG_COLS = BASE_ID_COLS + VALUE_COLS

# Quick sanity print (should be 30 total cols)
len(FINAL_LONG_COLS), FINAL_LONG_COLS[:8], FINAL_LONG_COLS[-8:]

(30,
 ['ts',
  'symbol',
  'open_time',
  'spot_close',
  'mark_close',
  'index_close',
  'premium_close',
  'basis_mark_vs_spot'],
 ['agg_taker_sell_trades',
  'agg_buy_notional',
  'agg_sell_notional',
  'agg_buy_vwap',
  'agg_sell_vwap',
  'agg_first_price',
  'agg_last_price',
  'agg_first_trade_id'])

In [ ]:
# Enumerate all joined monthly parquet paths (source of truth for final build)

# collect joined month part paths

from pathlib import Path  
import os  

joined_files = []
for sym in SYMBOLS:
    for (y, m) in YEAR_MONTHS:
        p = joined_month_path(Path(JOINED_MONTH_ROOT), interval=INTERVAL, symbol=sym, year=y, month=m)
        joined_files.append(p)

missing = [str(p) for p in joined_files if not p.exists()]
{"joined_files": len(joined_files), "missing": len(missing)}, missing[:5]

({'joined_files': 84, 'missing': 0}, [])

In [ ]:
# Build final long (single in-memory concat), enforce schema, save, read-back sanity check

# Read + concat
parts = []
for p in joined_files:
    df = pd.read_parquet(p, columns=FINAL_LONG_COLS)
    parts.append(df)

long_df = pd.concat(parts, ignore_index=True)

# Sort for stability (ts then symbol)
long_df = long_df.sort_values(["ts", "symbol"]).reset_index(drop=True)

# Save path
FINAL_LONG_PATH = (
    Path(FINAL_LONG_ROOT)
    / f"interval={INTERVAL}"
    / f"part-{START_UTC.strftime('%Y%m%d')}-{END_EXCL_UTC.strftime('%Y%m%d')}.parquet"
)
FINAL_LONG_PATH.parent.mkdir(parents=True, exist_ok=True)
long_df.to_parquet(FINAL_LONG_PATH, index=False)

# Read-back minimal check
long_head = pd.read_parquet(FINAL_LONG_PATH, columns=["ts","symbol"] + VALUE_COLS[:5]).head(3)
str(FINAL_LONG_PATH), long_df.shape, long_head


('C:\\Users\\quantbase\\Desktop\\marketdata\\norm\\_runs\\e2e_microagg_20260206\\master_long_plus_microaggtrades\\interval=15m\\part-20250101-20260101.parquet',
 (245280, 30),
                          ts    symbol      open_time  spot_close  \
 0 2025-01-01 00:00:00+00:00  ADA-USDT  1735689600000      0.8512   
 1 2025-01-01 00:00:00+00:00  BNB-USDT  1735689600000    704.0100   
 2 2025-01-01 00:00:00+00:00  BTC-USDT  1735689600000  93656.1800   
 
      mark_close   index_close  premium_close  
 0      0.850900      0.850975       0.000000  
 1    703.874597    703.924150      -0.000147  
 2  93637.200000  93650.139149      -0.000025  )

In [21]:
# Final long QC (structure + “micro columns must not be NaN”)

qc_long = {}

qc_long["rows"] = int(len(long_df))
qc_long["cols"] = int(long_df.shape[1])
qc_long["dups_ts_symbol"] = int(long_df.duplicated(["ts","symbol"]).sum())

# Expected rows
expected_per_symbol = int(len(pd.date_range(START_UTC, END_EXCL_UTC, freq=FREQ, inclusive="left")))
qc_long["expected_per_symbol"] = expected_per_symbol
qc_long["expected_total_rows"] = expected_per_symbol * len(SYMBOLS)

# Per-symbol ts uniqueness + counts
by_sym = long_df.groupby("symbol")["ts"].agg(["count", "nunique"]).rename(columns={"count":"rows","nunique":"ts_unique"})
qc_long["symbols"] = int(by_sym.shape[0])
qc_long["min_rows_per_symbol"] = int(by_sym["rows"].min())
qc_long["max_rows_per_symbol"] = int(by_sym["rows"].max())
qc_long["all_ts_unique_per_symbol"] = bool((by_sym["rows"] == by_sym["ts_unique"]).all())

# Micro NaN fractions (these must all be 0.0)
micro_cols = [c for c in VALUE_COLS if c.startswith("agg_") and c not in ("agg_sum_qty","agg_trades","agg_cvd_qty","agg_vwap","agg_last_trade_id")]
micro_nan = long_df[micro_cols].isna().mean().sort_values(ascending=False)
qc_long["micro_nan_max"] = float(micro_nan.max())
qc_long["micro_nan_nonzero_cols"] = micro_nan[micro_nan > 0].to_dict()

qc_long, by_sym.head(10), micro_nan.head(10)


({'rows': 245280,
  'cols': 30,
  'dups_ts_symbol': 0,
  'expected_per_symbol': 35040,
  'expected_total_rows': 245280,
  'symbols': 7,
  'min_rows_per_symbol': 35040,
  'max_rows_per_symbol': 35040,
  'all_ts_unique_per_symbol': True,
  'micro_nan_max': 0.0,
  'micro_nan_nonzero_cols': {}},
             rows  ts_unique
 symbol                     
 ADA-USDT   35040      35040
 BNB-USDT   35040      35040
 BTC-USDT   35040      35040
 ETH-USDT   35040      35040
 LINK-USDT  35040      35040
 SOL-USDT   35040      35040
 XRP-USDT   35040      35040,
 agg_taker_buy_qty        0.0
 agg_taker_sell_qty       0.0
 agg_taker_buy_trades     0.0
 agg_taker_sell_trades    0.0
 agg_buy_notional         0.0
 agg_sell_notional        0.0
 agg_buy_vwap             0.0
 agg_sell_vwap            0.0
 agg_first_price          0.0
 agg_last_price           0.0
 dtype: float64)

In [22]:
# Build final wide from final long, save, read-back sanity check

# Build wide: index ts, columns are flattened as col__symbol
wide = (
    long_df.set_index(["ts","symbol"])[VALUE_COLS]
    .unstack("symbol")
)

# Flatten multiindex columns: (value_col, symbol) -> "value_col__SYMBOL"
wide.columns = [f"{v}__{s}" for (v, s) in wide.columns]
wide = wide.sort_index().sort_index(axis=1)

FINAL_WIDE_PATH = (
    Path(FINAL_WIDE_ROOT)
    / f"interval={INTERVAL}"
    / f"part-{START_UTC.strftime('%Y%m%d')}-{END_EXCL_UTC.strftime('%Y%m%d')}.parquet"
)
FINAL_WIDE_PATH.parent.mkdir(parents=True, exist_ok=True)
wide.to_parquet(FINAL_WIDE_PATH, index=True)

wide_head = pd.read_parquet(FINAL_WIDE_PATH).head(3)
str(FINAL_WIDE_PATH), wide.shape, wide_head

('C:\\Users\\quantbase\\Desktop\\marketdata\\norm\\_runs\\e2e_microagg_20260206\\master_wide_plus_microaggtrades\\interval=15m\\part-20250101-20260101.parquet',
 (35040, 196),
                            agg_buy_notional__ADA-USDT  \
 ts                                                      
 2025-01-01 00:00:00+00:00                399716.02228   
 2025-01-01 00:15:00+00:00                226550.94634   
 2025-01-01 00:30:00+00:00                639436.92646   
 
                            agg_buy_notional__BNB-USDT  \
 ts                                                      
 2025-01-01 00:00:00+00:00                427074.04320   
 2025-01-01 00:15:00+00:00                250985.74655   
 2025-01-01 00:30:00+00:00                500322.25048   
 
                            agg_buy_notional__BTC-USDT  \
 ts                                                      
 2025-01-01 00:00:00+00:00                5.915902e+06   
 2025-01-01 00:15:00+00:00                4.905500e+06   
 2025-01

In [24]:
# Wide QC + reconciliation

wide_saved = pd.read_parquet(FINAL_WIDE_PATH)
wide_saved = wide_saved.sort_index().sort_index(axis=1)

qc_wide = {}
qc_wide["wide_rows"] = int(wide.shape[0])
qc_wide["wide_cols"] = int(wide.shape[1])
qc_wide["expected_rows"] = expected_per_symbol
qc_wide["expected_cols"] = len(VALUE_COLS) * len(SYMBOLS)

qc_wide["index_equal"] = bool(wide.index.equals(wide_saved.index))
qc_wide["cols_equal"] = bool(list(wide.columns) == list(wide_saved.columns))

# Numeric equality checks (should be exact; if float round-trip differences show up, we’ll switch to tolerance)
qc_wide["global_max_abs_diff"] = float((wide_saved - wide).abs().to_numpy().max())

# Check micro NaNs in wide
micro_wide_cols = [c for c in wide.columns if c.startswith("agg_") and ("__" in c) and (c.split("__")[0] in micro_cols)]
qc_wide["micro_nan_max_wide"] = float(wide[micro_wide_cols].isna().mean().max())

qc_wide

{'wide_rows': 35040,
 'wide_cols': 196,
 'expected_rows': 35040,
 'expected_cols': 196,
 'index_equal': True,
 'cols_equal': True,
 'global_max_abs_diff': 0.0,
 'micro_nan_max_wide': 0.0}

In [25]:
test_df = pd.read_parquet(FINAL_LONG_PATH)

In [27]:
test_df.to_csv(FINAL_LONG_PATH.with_suffix(".csv"), index=False)

In [28]:
# ----- Move file to norm/

import shutil  
import time  


# ====== CONFIG (assumes these already exist from earlier cells) ======
# NORM_ROOT: Path
# RUN_ROOT: Path
# MICRO_AGG_NORM_ROOT: Path
# JOINED_MONTH_ROOT: Path
# FINAL_LONG_ROOT: Path
# FINAL_WIDE_ROOT: Path
# SYMBOLS: list[str]
# YEAR_MONTHS: list[tuple[int,int]]
# INTERVAL: str  (e.g., "15m")

assert isinstance(NORM_ROOT, Path) and NORM_ROOT.exists()
assert isinstance(RUN_ROOT, Path) and RUN_ROOT.exists()

BACKUP_ROOT = NORM_ROOT / "_backup_promotions"
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

ts_tag = time.strftime("%Y%m%d_%H%M%S")


def _backup_if_exists(dst: Path) -> None:
    if dst.exists():
        bkp = BACKUP_ROOT / f"{dst.name}__backup_{ts_tag}"
        print(f"[backup] {dst} -> {bkp}")
        shutil.move(str(dst), str(bkp))


def _copy_tree(src: Path, dst: Path) -> None:
    if not src.exists():
        raise FileNotFoundError(f"Missing source: {src}")
    _backup_if_exists(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    print(f"[copy] {src} -> {dst}")
    shutil.copytree(src, dst)


def _count_month_parts(root: Path, interval: str) -> int:
    # expects layout: root/interval=15m/year=YYYY/month=MM/symbol=SYM/part-YYYY-MM.parquet
    return len(list(root.glob(f"interval={interval}/year=*/month=*/symbol=*/part-*.parquet")))


def _count_parquets(root: Path) -> int:
    return len(list(root.rglob("*.parquet")))


def _verify_monthly(root_src: Path, root_dst: Path, interval: str, expected_parts: int) -> None:
    n_src = _count_month_parts(root_src, interval)
    n_dst = _count_month_parts(root_dst, interval)
    print(f"[verify-monthly] src_parts={n_src} dst_parts={n_dst} expected={expected_parts}")
    assert n_src == expected_parts, {"where": "src", "got": n_src, "expected": expected_parts, "root": str(root_src)}
    assert n_dst == expected_parts, {"where": "dst", "got": n_dst, "expected": expected_parts, "root": str(root_dst)}


def _verify_all_parquets(root_src: Path, root_dst: Path) -> None:
    n_src = _count_parquets(root_src)
    n_dst = _count_parquets(root_dst)
    print(f"[verify-tree] src_parquets={n_src} dst_parquets={n_dst}")
    assert n_src == n_dst, {"src": n_src, "dst": n_dst, "src_root": str(root_src), "dst_root": str(root_dst)}


# ====== DESTINATION CANONICAL ROOTS ======
CANON_MICRO_AGG_ROOT = NORM_ROOT / "spot_aggtrades_resampled_micro"
CANON_JOINED_MONTH_ROOT = NORM_ROOT / "master_plus_microaggtrades"
CANON_FINAL_LONG_ROOT = NORM_ROOT / "master_long_plus_microaggtrades"
CANON_FINAL_WIDE_ROOT = NORM_ROOT / "master_wide_plus_microaggtrades"

expected_parts = len(SYMBOLS) * len(YEAR_MONTHS)
print({"expected_month_parts": expected_parts, "symbols": len(SYMBOLS), "months": len(YEAR_MONTHS)})

# ====== COPY ======
_copy_tree(MICRO_AGG_NORM_ROOT, CANON_MICRO_AGG_ROOT)
_copy_tree(JOINED_MONTH_ROOT, CANON_JOINED_MONTH_ROOT)
_copy_tree(FINAL_LONG_ROOT, CANON_FINAL_LONG_ROOT)
_copy_tree(FINAL_WIDE_ROOT, CANON_FINAL_WIDE_ROOT)

# ====== VERIFY ======
_verify_monthly(MICRO_AGG_NORM_ROOT, CANON_MICRO_AGG_ROOT, INTERVAL, expected_parts)
_verify_monthly(JOINED_MONTH_ROOT, CANON_JOINED_MONTH_ROOT, INTERVAL, expected_parts)

_verify_all_parquets(FINAL_LONG_ROOT, CANON_FINAL_LONG_ROOT)
_verify_all_parquets(FINAL_WIDE_ROOT, CANON_FINAL_WIDE_ROOT)

print("[done] promoted run artifacts into norm/")

{'expected_month_parts': 84, 'symbols': 7, 'months': 12}
[backup] C:\Users\quantbase\Desktop\marketdata\norm\spot_aggtrades_resampled_micro -> C:\Users\quantbase\Desktop\marketdata\norm\_backup_promotions\spot_aggtrades_resampled_micro__backup_20260227_191050
[copy] C:\Users\quantbase\Desktop\marketdata\norm\_runs\e2e_microagg_20260206\spot_aggtrades_resampled_micro -> C:\Users\quantbase\Desktop\marketdata\norm\spot_aggtrades_resampled_micro
[backup] C:\Users\quantbase\Desktop\marketdata\norm\master_plus_microaggtrades -> C:\Users\quantbase\Desktop\marketdata\norm\_backup_promotions\master_plus_microaggtrades__backup_20260227_191050
[copy] C:\Users\quantbase\Desktop\marketdata\norm\_runs\e2e_microagg_20260206\master_plus_microaggtrades -> C:\Users\quantbase\Desktop\marketdata\norm\master_plus_microaggtrades
[backup] C:\Users\quantbase\Desktop\marketdata\norm\master_long_plus_microaggtrades -> C:\Users\quantbase\Desktop\marketdata\norm\_backup_promotions\master_long_plus_microaggtrades_